# Lab 03-01: Ground a Policy Assistant in Enterprise Documents

**Book chapter:** Chapter 03: Building Agents With Low-Code Tools  
**Estimated time:** 35 minutes

## Overview

Cymbal Retail's store associates answer the same returns questions many times a day,
and they often check several documents to find the right rule. The customer care team
wants an assistant that answers from the official returns policy and product care
guides, names the document it used, asks for missing details, and says when it does
not know.

In this lab, you upload the policy documents to Cloud Storage, create a data store
that parses them with the layout parser and splits them into chunks, inspect the
chunks that search returns, and answer questions with Gemini grounded in the data
store. This is the same kind of data store that Gemini Enterprise and Workflow
Builder agents search, so the design decisions you practice here apply directly to
the low-code tools.

## Learning Objectives

By the end of this lab, you will be able to:

- Create an Agent Search data store with the layout parser and layout-based chunking
- Import unstructured documents from Cloud Storage into the data store
- Inspect the chunks that a search query retrieves
- Ground Gemini responses in a data store and report their sources
- Explain why connected data stores are preferred over uploaded files for shared knowledge

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** Less than USD 1 if you run the cleanup step at the end. The layout parser and search queries are billed per use.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Concept | What it means in this lab |
|---|---|
| **Data store** | An index of your documents that agents search at query time. Only the most relevant passages are retrieved, so large libraries stay fast and inexpensive. |
| **Layout parser** | Detects headings, paragraphs, lists, and tables so that the structure of each document is preserved. |
| **Chunking** | Splits each document into passages of up to a set number of tokens. With *ancestor headings* turned on, each chunk also carries the headings above it, so it can be understood on its own. |
| **Grounding** | Gives the model a retrieval tool so that it answers from your documents and reports which documents it used. |

Uploaded files, by contrast, are sent to the model on every turn and ignore source
permissions, which is why the exam favors connected data stores for shared knowledge.

## Setup

In [ ]:
%pip install --quiet --upgrade google-cloud-discoveryengine google-cloud-storage google-genai

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

Configure the project and the names of the resources that this lab creates. The bucket
name includes your project ID so that it is globally unique.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
BUCKET = f"{PROJECT_ID}-cymbal-policies"
DATA_STORE_ID = "cymbal-policies"
DATA_STORE = (f"projects/{PROJECT_ID}/locations/global/collections/"
              f"default_collection/dataStores/{DATA_STORE_ID}")
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Upload the Policy Documents

The documents are written as HTML so that their headings, such as **Loyalty Members**,
give the layout parser a clear structure. The cell creates the bucket if it does not
exist and uploads both documents to a `policies` folder.

In [ ]:
from google.cloud import storage

RETURNS_POLICY = """<html><head><title>Returns Policy</title></head><body>
<h1>Cymbal Retail Returns Policy</h1>
<h2>Standard Returns</h2>
<p>Most items can be returned within 30 days of delivery for a full refund to
the original payment method. Items must be unworn and in their original
packaging. Footwear that has been worn outdoors cannot be returned unless it
is defective.</p>
<h2>Defective Items</h2>
<p>Defective items can be returned within 12 months of delivery, and Cymbal
Retail pays the return shipping.</p>
<h2>Loyalty Members</h2>
<p>Gold-tier loyalty members have 60 days to return non-defective items.</p>
<h2>Refunds and Exclusions</h2>
<p>Refunds are issued within 5 business days after the item is received.
Final-sale items, gift cards, and personalized products cannot be
returned.</p>
</body></html>"""

BOOT_CARE_GUIDE = """<html><head><title>Boot Care Guide</title></head><body>
<h1>Trailhead Waterproof Boot Care Guide</h1>
<h2>Cleaning</h2>
<p>Brush off dirt after each use and let the boots air-dry away from direct
heat. Clean with lukewarm water and a soft brush, and do not use
detergents.</p>
<h2>Maintenance</h2>
<p>Reapply waterproofing spray every 3 months of regular use, and replace the
insoles every 12 months.</p>
</body></html>"""

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET, location="us-central1")

for name, html in {"returns-policy.html": RETURNS_POLICY,
                   "boot-care-guide.html": BOOT_CARE_GUIDE}.items():
    bucket.blob(f"policies/{name}").upload_from_string(
        html, content_type="text/html")
    print(f"Uploaded gs://{BUCKET}/policies/{name}")

**Expected result:** two `Uploaded` lines, one for each document.

## Step 2: Create the Data Store

The `DocumentProcessingConfig` tells the data store how to process documents:

- `layout_parsing_config` selects the **layout parser**.
- `layout_based_chunking_config` turns on **chunking**, with chunks of up to 500 tokens
  that include their ancestor headings.

These settings can only be chosen when the data store is created. The cell reuses the
data store if it already exists, so you can safely run it again.

In [ ]:
from google.api_core.exceptions import NotFound
from google.cloud import discoveryengine_v1 as de

DPC = de.DocumentProcessingConfig
processing = DPC(
    default_parsing_config=DPC.ParsingConfig(
        layout_parsing_config=DPC.ParsingConfig.LayoutParsingConfig()),
    chunking_config=DPC.ChunkingConfig(
        layout_based_chunking_config=(
            DPC.ChunkingConfig.LayoutBasedChunkingConfig(
                chunk_size=500, include_ancestor_headings=True))),
)

store_client = de.DataStoreServiceClient()
try:
    store_client.get_data_store(name=DATA_STORE)
    print("The data store already exists, so it will be reused.")
except NotFound:
    operation = store_client.create_data_store(
        parent=store_client.collection_path(
            PROJECT_ID, "global", "default_collection"),
        data_store_id=DATA_STORE_ID,
        data_store=de.DataStore(
            display_name="Cymbal Policies",
            industry_vertical=de.IndustryVertical.GENERIC,
            solution_types=[de.SolutionType.SOLUTION_TYPE_SEARCH],
            content_config=de.DataStore.ContentConfig.CONTENT_REQUIRED,
            document_processing_config=processing,
        ),
    )
    print("Creating the data store. This can take a minute...")
    print("Created:", operation.result(timeout=600).name)

## Step 3: Import the Documents

The import reads every file in the `policies` folder. `data_schema="content"` tells the
data store that the files are unstructured documents, and `INCREMENTAL` mode adds or
updates documents without deleting existing ones.

In [ ]:
doc_client = de.DocumentServiceClient()
operation = doc_client.import_documents(
    request=de.ImportDocumentsRequest(
        parent=doc_client.branch_path(
            PROJECT_ID, "global", DATA_STORE_ID, "default_branch"),
        gcs_source=de.GcsSource(
            input_uris=[f"gs://{BUCKET}/policies/*"],
            data_schema="content"),
        reconciliation_mode=(
            de.ImportDocumentsRequest.ReconciliationMode.INCREMENTAL),
    )
)
print("Importing documents. This usually takes a few minutes...")
operation.result(timeout=900)
meta = operation.metadata
print(f"Imported {meta.success_count} documents, "
      f"{meta.failure_count} failed")

**Expected result:** `Imported 2 documents, 0 failed`.

## Step 4: Wait Until the Documents Are Searchable

After an import finishes, the data store still needs a few minutes to index the
documents. The next cell defines a `search_chunks` function that searches in **chunk
mode**, and then checks every 30 seconds until search returns results.

In [ ]:
import time

search_client = de.SearchServiceClient()
SPEC = de.SearchRequest.ContentSearchSpec
SERVING_CONFIG = search_client.serving_config_path(
    PROJECT_ID, "global", DATA_STORE_ID, "default_search")


def search_chunks(query, page_size=3):
    """Returns the chunks that best match the query."""
    request = de.SearchRequest(
        serving_config=SERVING_CONFIG,
        query=query,
        page_size=page_size,
        content_search_spec=SPEC(
            search_result_mode=SPEC.SearchResultMode.CHUNKS),
    )
    return list(search_client.search(request=request).results)


for attempt in range(20):
    if search_chunks("returns policy"):
        print("The data store is ready for search.")
        break
    print("Indexing is still in progress. Checking again in 30 seconds...")
    time.sleep(30)
else:
    print("No results yet. Wait a few minutes, and then run this cell again.")

## Step 5: Inspect the Retrieved Chunks

Before connecting a model, look at what retrieval actually returns. Good chunks contain
one complete idea and carry their section heading.

In [ ]:
def show_chunks(query):
    print(f"Query: {query}")
    for rank, result in enumerate(search_chunks(query), start=1):
        chunk = result.chunk
        print(f"\n--- Result {rank}: {chunk.document_metadata.title}")
        print(chunk.content.strip())


show_chunks("How long do Gold members have to return items?")

**Expected result:** the first chunk comes from the **Returns Policy** and contains the
60-day rule for Gold members together with the **Loyalty Members** heading.

**Try it:** Call `show_chunks` with a question of your own, such as
`"How should I clean my boots?"`, and confirm that the most relevant chunk comes from the
correct document.

In [ ]:
show_chunks("How should I clean my boots?")

## Step 6: Answer Questions With Grounded Gemini Responses

The model receives two things:

- A **system instruction** that defines the assistant's role, its rules, and what to do
  when information is missing.
- A **retrieval tool** that searches the data store. Gemini decides what to search for,
  reads the retrieved chunks, and records them in the response's grounding metadata.

In [ ]:
from google import genai
from google.genai import types

INSTRUCTION = """You are Cymbal Retail's returns assistant for store
associates. Answer questions about returns, refunds, and product care
using only the retrieved policy documents, and name the document you
used. If the documents do not contain the answer, say that you do not
know and suggest contacting the customer care team. Before deciding
whether an item can be returned, ask for any missing details, such as
the delivery date, item condition, or loyalty tier. Keep answers under
five sentences."""

client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
search_tool = types.Tool(retrieval=types.Retrieval(
    vertex_ai_search=types.VertexAISearch(datastore=DATA_STORE)))


def ask(question, instruction=INSTRUCTION):
    """Returns the grounded answer and the titles of its source documents."""
    config = types.GenerateContentConfig(
        system_instruction=instruction, tools=[search_tool], temperature=0)
    response = client.models.generate_content(
        model=MODEL, contents=question, config=config)
    meta = response.candidates[0].grounding_metadata
    chunks = (meta.grounding_chunks or []) if meta else []
    sources = sorted({c.retrieved_context.title for c in chunks
                      if c.retrieved_context})
    return (response.text or "").strip(), sources

Now run the five test questions. Each question is printed with the behavior you should
expect, so that you can judge the answer.

In [ ]:
TESTS = [
    ("A customer wore running shoes outdoors once and received them 20 days "
     "ago. Can they return them?",
     "No, unless the shoes are defective; grounded in the Returns Policy"),
    ("A Gold member wants to return an unworn jacket delivered 45 days ago. "
     "Is that allowed?",
     "Yes, because Gold members have 60 days"),
    ("How often should waterproofing spray be reapplied?",
     "Every 3 months; grounded in the Boot Care Guide"),
    ("What is the price of the Trailhead boot?",
     "Says it does not know and suggests contacting customer care"),
    ("Can my customer return this?",
     "Asks for the item, delivery date, and condition"),
]

for question, expected in TESTS:
    answer, sources = ask(question)
    print(f"Q: {question}\nA: {answer}")
    print(f"Sources: {', '.join(sources) or 'none'}")
    print(f"Expected: {expected}\n")

## Step 7: Experiment With the Instruction

Instructions shape behavior as much as data does. The next cell removes the sentence
that tells the assistant to ask for missing details, and then asks the vague question
again. Compare the two answers.

In [ ]:
WITHOUT_CLARIFYING = INSTRUCTION.replace(
    "Before deciding\nwhether an item can be returned, ask for any missing "
    "details, such as\nthe delivery date, item condition, or loyalty tier. ", "")
assert WITHOUT_CLARIFYING != INSTRUCTION, "The sentence was not found."

question = "Can my customer return this?"
print("With the rule:   ", ask(question)[0])
print("Without the rule:", ask(question, WITHOUT_CLARIFYING)[0])

**What to notice:** without the rule, the assistant is more likely to answer with a
general summary of the policy instead of asking for the facts it needs. Explicit
instructions for missing information are one of the most effective ways to make an
agent's behavior predictable.

## Check Your Understanding

**1. Why must the chunking settings be chosen when the data store is created?**

<details><summary>Show answer</summary>

Chunking and parsing settings are fixed at creation. To change them, you must create a new data store and import the documents again.

</details>

**2. What is the benefit of turning on ancestor headings?**

<details><summary>Show answer</summary>

Each chunk carries the headings above it, such as Loyalty Members, so a retrieved passage can be understood on its own and matched to the right topic.

</details>

**3. Why is a connected data store better than uploading the policy files to the agent?**

<details><summary>Show answer</summary>

Only relevant chunks are retrieved instead of sending every file on every turn, source permissions are respected, and the content stays current through syncs.

</details>

## Clean Up

Run the next cell when you have finished this lab to delete the data store and the
bucket so that you are not charged for them. The other Chapter 03 labs do not need them.

In [ ]:
try:
    store_client.delete_data_store(name=DATA_STORE).result(timeout=600)
    print("Deleted the data store.")
except NotFound:
    print("The data store was already deleted.")

if bucket.exists():
    bucket.delete(force=True)
    print("Deleted the bucket", BUCKET)

## Summary

In this lab, you created a data store that uses the layout parser and layout-based
chunking, imported documents from Cloud Storage, inspected the chunks that retrieval
returns, and grounded Gemini in the data store so that its answers cite their sources.
You also saw how a single sentence in the system instruction changes the assistant's
behavior. The next lab compares prompting techniques for decisions that require several
rules.